# Positioning and forced flows, CSI 300 - 2026-09-29

**Question:** where are mechanical, forced flows likely to concentrate - option-dealer hedging around big strikes, index-futures hedgers adding shorts, and margin-financing liquidations - and what do they say after the 2026-09-24 / 09-28 sell-off?

All calculation comes from `quant_cn.calculation` (`GammaExposureCalculator`, `FuturesPositioning`, `MarginLadderEstimator`; tests in `tests/calculation/`). Plan: `plans/execution/09_positioning_flows.md`.

**Assumptions (each a logged decision, none validated yet):**
- Dealer gamma sign: dealers **long calls, short puts** (customers write calls and buy puts) - D-057. Tushare has open interest only, not who holds it, so the sign is a model.
- Futures top-20 positions are gross broker aggregates (代客 = client accounts), not client intent - D-059.
- Margin vintages are first-in-first-out at the entry close; forced sale at `entry × 1.3 / (1 + 1.0)` (maintenance 130%, own capital = debt) - D-058. Cross-collateral and per-account detail are not visible.

These levels are **hypotheses** until plan 09's event study shows moves accelerating through them.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from quant_cn.calculation.black76_model import Black76Model
from quant_cn.calculation.forward_price_estimator import ForwardPriceEstimator
from quant_cn.calculation.futures_positioning import FuturesPositioning
from quant_cn.calculation.gamma_exposure_calculator import GammaExposureCalculator
from quant_cn.calculation.margin_ladder_estimator import MarginLadderEstimator
from quant_cn.calculation.option_chain_builder import OptionChainBuilder
from quant_cn.calculation.risk_free_curve import RiskFreeCurve
from quant_cn.core.config import Config
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.visualization.chart_theme import ChartTheme

DAY, PREV = "20260928", "20260924"
config = Config.load()
catalog = LakeCatalog(config.lake.root, config.datasets, read_only=True)
query = LakeQuery(catalog)
theme = ChartTheme.from_mode("light")
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
spot = query.sql("SELECT trade_date, close FROM curated.index_daily WHERE ts_code = '000300.SH' ORDER BY 1").set_index("trade_date")["close"]
print(f"CSI 300: {PREV} {spot[PREV]:.1f} -> {DAY} {spot[DAY]:.1f} ({(spot[DAY] / spot[PREV] - 1) * 100:+.2f}%)")


def style(fig, title, y_title, height=440):
    fig.update_layout(title=title, paper_bgcolor=theme.surface, plot_bgcolor=theme.surface,
                      font={"color": theme.text_primary}, width=1000, height=height, hovermode="x unified",
                      margin={"l": 70, "r": 40, "t": 56, "b": 40}, legend={"orientation": "h", "y": -0.18})
    fig.update_xaxes(gridcolor=theme.grid, tickfont={"color": theme.text_secondary})
    fig.update_yaxes(title_text=y_title, gridcolor=theme.grid, tickfont={"color": theme.text_secondary})
    return fig

CSI 300: 20260924 4439.1 -> 20260928 4340.8 (-2.22%)


## 0. Data reconciliation (plan 09 phase 1 done criterion)

Top-20 broker holdings can only cover part of total open interest, so the check is the coverage share per product (stable, well below 100%); per-stock margin balances must sum to the exchange aggregates.

In [2]:
fut_oi = query.sql(
    "SELECT trade_date, regexp_extract(ts_code, '^([A-Z]+)', 1) AS product, sum(oi) AS oi FROM derived.fut_daily_one_sided "
    "WHERE regexp_matches(ts_code, '^(IF|IH|IC|IM)[0-9]{4}') AND trade_date >= '20250101' GROUP BY 1, 2")
top20 = query.sql(
    "SELECT trade_date, regexp_extract(symbol, '^([A-Z]+)', 1) AS product, sum(long_hld) AS long_top20, sum(short_hld) AS short_top20 "
    "FROM derived.fut_holding_brokers WHERE regexp_matches(symbol, '^(IF|IH|IC|IM)[0-9]{4}$') AND trade_date >= '20250101' GROUP BY 1, 2")
recon = fut_oi.merge(top20, on=["trade_date", "product"])
recon["long_share"] = recon["long_top20"] / recon["oi"]
recon["short_share"] = recon["short_top20"] / recon["oi"]
print(recon.groupby("product")[["long_share", "short_share"]].agg(["min", "median", "max"]).round(3).to_string())
margin_check = query.sql(
    "SELECT m.trade_date, m.rzye AS exchange_rzye, d.rzye AS detail_rzye FROM "
    "(SELECT trade_date, sum(rzye) rzye FROM curated.margin WHERE exchange_id IN ('SSE', 'SZSE') GROUP BY 1) m "
    "JOIN (SELECT trade_date, sum(rzye) rzye FROM curated.margin_detail GROUP BY 1) d USING (trade_date)")
margin_check["gap_pct"] = (margin_check["detail_rzye"] / margin_check["exchange_rzye"] - 1) * 100
print(f"margin_detail vs SSE+SZSE aggregate: {len(margin_check)} days, gap % min {margin_check['gap_pct'].min():.3f}, "
      f"median {margin_check['gap_pct'].median():.3f}, max {margin_check['gap_pct'].max():.3f}")

        long_share               short_share              
               min median    max         min median    max
product                                                   
IC           0.679  0.729  0.797       0.774  0.816  0.845
IF           0.655  0.723  0.772       0.786  0.833  0.867
IH           0.593  0.687  0.762       0.720  0.853  0.917
IM           0.647  0.699  0.733       0.764  0.807  0.842
margin_detail vs SSE+SZSE aggregate: 422 days, gap % min -0.694, median -0.373, max -0.112


## A. Option-dealer gamma around big strikes (CSI 300 index options, IO)

In [3]:
basic = query.sql("SELECT * FROM curated.opt_basic WHERE opt_code = 'OP000300.SH'")
daily = query.sql("SELECT * FROM curated.opt_daily WHERE trade_date IN (?, ?) AND ts_code IN (SELECT ts_code FROM curated.opt_basic WHERE opt_code = 'OP000300.SH')", [PREV, DAY])
builder = OptionChainBuilder()
chain = builder.build_chain(basic, daily, "OP000300.SH")
gex_calc = GammaExposureCalculator(RiskFreeCurve(query.sql("SELECT * FROM curated.shibor")), builder,
                                   ForwardPriceEstimator(), Black76Model(), max_days=90)
exposure = gex_calc.compute(chain[chain["trade_date"] == DAY], DAY, multiplier=100)
exposure_prev = gex_calc.compute(chain[chain["trade_date"] == PREV], PREV, multiplier=100)
total, total_prev = exposure["gex"].sum(), exposure_prev["gex"].sum()
print(f"total dealer GEX: {PREV} {total_prev / 1e9:+.2f} bn CNY per 1%  ->  {DAY} {total / 1e9:+.2f} bn CNY per 1%")
by_strike = exposure.pivot_table(index="strike", columns="call_put", values=["oi", "gex"], aggfunc="sum").fillna(0)
by_strike[("gex", "net")] = by_strike[("gex", "C")] + by_strike[("gex", "P")]
oi_prev = exposure_prev.pivot_table(index="strike", columns="call_put", values="oi", aggfunc="sum").fillna(0)
for side in ("C", "P"):
    by_strike[("d_oi", side)] = by_strike[("oi", side)] - oi_prev[side].reindex(by_strike.index).fillna(0)
near = by_strike[(by_strike.index >= spot[DAY] * 0.9) & (by_strike.index <= spot[DAY] * 1.1)]
(near[[("oi", "P"), ("oi", "C"), ("d_oi", "P"), ("d_oi", "C"), ("gex", "net")]] / [1, 1, 1, 1, 1e6]).round(1).rename(
    columns={"gex": "gex_mn_cny"}, level=0)

total dealer GEX: 20260924 +0.30 bn CNY per 1%  ->  20260928 +0.10 bn CNY per 1%


oi            d_oi         gex_mn_cny
call_put       P        C      P       C        net
strike                                             
4000.0    3515.0    350.0  802.0    28.0      -38.5
4050.0    2819.0    113.0  223.0    31.0      -38.6
4100.0    4847.0    533.0  169.0   169.0      -74.8
4150.0    2723.0    127.0   56.0    75.0      -57.6
4200.0    4981.0   1274.0  245.0   521.0      -87.6
4250.0    3212.0    542.0  575.0   285.0      -92.8
4300.0    6127.0   3674.0  504.0  1463.0      -84.3
4350.0    4439.0   2537.0  557.0  1451.0      -79.5
4400.0    8961.0   6892.0  -48.0  2130.0      -92.7
4450.0    5705.0   5972.0 -493.0  1192.0        8.7
4500.0    6825.0  15721.0 -657.0  2761.0      224.6
4550.0    2755.0   9241.0 -852.0   505.0      141.0
4600.0    3241.0  12370.0 -109.0   641.0      147.4
4650.0     806.0   5625.0  -73.0    38.0       60.0
4700.0    1587.0   8144.0  -82.0  -328.0       67.6
4750.0     318.0   2543.0  -40.0  -342.0       17.0

In [4]:
levels = np.linspace(spot[DAY] * 0.88, spot[DAY] * 1.12, 49)
f_near = float(exposure.sort_values("days")["forward"].iloc[0])
profile = gex_calc.compute_profile(exposure, levels * f_near / spot[DAY])
profile["index_level"] = levels
flip_forward = gex_calc.find_flip(profile)
flip_index = None if flip_forward is None else flip_forward * spot[DAY] / f_near
print("gamma flip (index terms):", "none on the grid" if flip_index is None else f"{flip_index:.0f}", f"| spot {spot[DAY]:.0f}")
fig = go.Figure()
fig.add_bar(x=by_strike.index, y=by_strike[("gex", "net")] / 1e6, name="net dealer GEX by strike",
            marker={"color": np.where(by_strike[("gex", "net")] >= 0, BLUE, ORANGE)},
            hovertemplate="strike %{x}: %{y:.0f} mn CNY per 1%<extra></extra>")
fig.add_vline(x=float(spot[DAY]), line={"color": theme.text_secondary, "dash": "dash"})
fig.add_annotation(x=float(spot[DAY]), y=1, yref="paper", text=f"CSI 300 {spot[DAY]:.0f}", showarrow=False,
                   xanchor="left", xshift=4, font={"color": theme.text_secondary})
style(fig, f"IO dealer gamma exposure by strike, {DAY} (blue: dampening, orange: amplifying)", "mn CNY per 1% move").show()
fig = go.Figure()
fig.add_scatter(x=profile["index_level"], y=profile["gex"] / 1e9, mode="lines", line={"color": BLUE, "width": 2},
                name="total dealer GEX", hovertemplate="index %{x:.0f}: %{y:.2f} bn<extra></extra>")
fig.add_hline(y=0, line={"color": theme.text_secondary, "width": 1})
fig.add_vline(x=float(spot[DAY]), line={"color": theme.text_secondary, "dash": "dash"})
if flip_index is not None:
    fig.add_vline(x=flip_index, line={"color": ORANGE, "dash": "dot"})
    fig.add_annotation(x=flip_index, y=0.95, yref="paper", text=f"flip {flip_index:.0f}", showarrow=False,
                       xanchor="right", xshift=-4, font={"color": theme.text_secondary})
style(fig, "IO total dealer GEX if the index moved (implied vols held)", "bn CNY per 1% move").show()

gamma flip (index terms): 4321 | spot 4341


**Reading it:** under the stated sign, negative GEX means dealers sell as the index falls and buy as it rises (amplifying); below the flip level hedging accelerates moves. The strike table shows where open interest was added on 09-28.

## B. Index-futures positioning (CFFEX)

In [5]:
fut_daily = query.sql("SELECT * FROM derived.fut_daily_one_sided WHERE trade_date >= '20250101'")
holding = query.sql("SELECT * FROM derived.fut_holding_brokers WHERE trade_date >= '20250101'")
index_spot = {"IF": "000300.SH", "IH": "000016.SH", "IC": "000905.SH", "IM": "000852.SH"}
closes = query.sql("SELECT ts_code, trade_date, close FROM curated.index_daily WHERE trade_date >= '20250101'")
positioning = {p: FuturesPositioning(p).compute(fut_daily, holding, closes[closes["ts_code"] == code].set_index("trade_date")["close"])
               for p, code in index_spot.items()}
summary = pd.DataFrame({p: df.set_index("trade_date").loc[DAY, ["price_change", "oi_change", "regime", "top20_net", "basis_pct", "basis_annualised"]]
                        for p, df in positioning.items()}).T
summary

,price_change,oi_change,regime,top20_net,basis_pct,basis_annualised
IF,-94.2,20941.0,new_shorts,-31290.0,-1.381211,-6.223977
IH,-34.4,7706.0,new_shorts,-16914.0,-1.308951,-5.898361
IC,-181.6,14808.0,new_shorts,-25466.0,-1.787217,-8.05351
IM,-233.8,24704.0,new_shorts,-60103.0,-1.877848,-8.461908


In [6]:
last = pd.concat([df.assign(product=p) for p, df in positioning.items()]).pivot(index="trade_date", columns="product", values="regime")
last.tail(8)

product,IC,IF,IH,IM
trade_date,,,,
20260916,short_covering,short_covering,short_covering,new_longs
20260917,long_liquidation,long_liquidation,long_liquidation,long_liquidation
20260918,short_covering,short_covering,short_covering,short_covering
20260921,short_covering,short_covering,new_longs,short_covering
20260922,new_shorts,new_longs,short_covering,new_longs
20260923,long_liquidation,long_liquidation,long_liquidation,long_liquidation
20260924,new_shorts,new_shorts,new_shorts,new_shorts
20260928,new_shorts,new_shorts,new_shorts,new_shorts


In [7]:
fig = go.Figure()
for p, color in (("IF", BLUE), ("IC", ORANGE), ("IM", AQUA)):
    df = positioning[p]
    fig.add_scatter(x=pd.to_datetime(df["trade_date"]), y=df["top20_net"] / 1e3, name=f"{p} top-20 net (k contracts)",
                    mode="lines", line={"color": color, "width": 2})
style(fig, "Top-20 broker net positions (long - short), 2025-01 to 2026-09-28", "thousand contracts").show()
fig = go.Figure()
for p, color in (("IF", BLUE), ("IC", ORANGE), ("IM", AQUA)):
    df = positioning[p]
    fig.add_scatter(x=pd.to_datetime(df["trade_date"]), y=df["basis_annualised"], name=f"{p} main-contract basis, annualised %",
                    mode="lines", line={"color": color, "width": 2})
fig.add_hline(y=0, line={"color": theme.text_secondary, "width": 1})
style(fig, "Futures basis to the index, annualised (negative = discount = hedging demand)", "% per year").show()

In [8]:
FuturesPositioning("IF").list_brokers(holding, DAY).head(8)

,broker,long,short,net,long_chg,short_chg
5,中信期货(代客),36693.0,61535.0,-24842.0,5429.0,5565.0
15,华泰期货(代客),10230.0,27926.0,-17696.0,1055.0,1176.0
30,招商期货(代客),1149.0,8784.0,-7635.0,-86.0,55.0
10,中银期货(代客),1688.0,6028.0,-4340.0,474.0,292.0
19,国信期货(代客),5986.0,10184.0,-4198.0,1526.0,994.0
9,中金财富(代客),9456.0,13402.0,-3946.0,-89.0,245.0
3,东证期货(代客),11829.0,14836.0,-3007.0,835.0,1229.0
17,华闻期货(代客),2063.0,5061.0,-2998.0,525.0,1091.0


## C. Margin financing: balances and a liquidation ladder

In [9]:
margin = query.sql("SELECT trade_date, sum(rzye) AS rzye, sum(rzmre) AS buy, sum(rzche) AS repay FROM curated.margin GROUP BY 1 ORDER BY 1")
margin["net_new_debt"] = margin["buy"] - margin["repay"]
print(margin.tail(5).assign(rzye_bn=lambda d: d["rzye"] / 1e9, net_bn=lambda d: d["net_new_debt"] / 1e9)[["trade_date", "rzye_bn", "net_bn"]].round(1).to_string(index=False))
fig = go.Figure()
fig.add_scatter(x=pd.to_datetime(margin["trade_date"]), y=margin["rzye"] / 1e12, mode="lines", line={"color": BLUE, "width": 2},
                name="financing balance (trn CNY)")
style(fig, "Margin financing balance, all exchanges", "trillion CNY", height=360).show()

trade_date  rzye_bn  net_bn
  20260921   2623.0    14.2
  20260922   2626.4     2.8
  20260923   2625.3    -1.0
  20260924   2607.9   -17.4
  20260928   2581.7   -26.1


In [10]:
detail = query.sql("SELECT ts_code, trade_date, rzye, rzmre, rzche FROM curated.margin_detail")
stock_closes = query.sql("SELECT ts_code, trade_date, close FROM curated.daily WHERE trade_date >= (SELECT min(trade_date) FROM curated.margin_detail)")
estimator = MarginLadderEstimator(maintenance_ratio=1.3, initial_margin_ratio=1.0)
ladder = estimator.compute_ladder(detail, stock_closes, DAY)
curve = estimator.compute_curve(ladder, list(range(0, 41, 5)))
print(f"financing debt covered: {ladder['debt'].sum() / 1e12:.2f} trn CNY across {ladder['ts_code'].nunique()} stocks; "
      f"seeded (pre-window) share {ladder.loc[ladder['seeded'] == 1, 'debt'].sum() / ladder['debt'].sum():.0%}")
curve.assign(debt_forced_bn=lambda d: d["debt_forced"] / 1e9, share=lambda d: (d["share_of_debt"] * 100).round(1))[["drawdown_pct", "debt_forced_bn", "share"]].round(1)

financing debt covered: 2.55 trn CNY across 4033 stocks; seeded (pre-window) share 0%


,drawdown_pct,debt_forced_bn,share
0,0.0,2.8,0.1
1,5.0,5.7,0.2
2,10.0,8.2,0.3
3,15.0,15.2,0.6
4,20.0,35.8,1.4
5,25.0,126.6,5.0
6,30.0,504.4,19.8
7,35.0,1953.2,76.7
8,40.0,2457.2,96.5


In [11]:
fig = go.Figure()
fig.add_scatter(x=curve["drawdown_pct"], y=curve["debt_forced"] / 1e9, mode="lines+markers", line={"color": ORANGE, "width": 2},
                marker={"size": 8}, name="debt at forced-sale price", hovertemplate="-%{x}%: %{y:.0f} bn CNY<extra></extra>")
style(fig, f"Estimated financing debt reaching its forced-sale price if every stock fell by x% from {DAY}", "bn CNY").show()
near_liq = ladder[ladder["distance_pct"] > -15].groupby("ts_code")["debt"].sum().sort_values(ascending=False).head(10) / 1e8
near_liq.rename("debt within 15% of forced sale (100m CNY)").round(1).to_frame()

,debt within 15% of forced sale (100m CNY)
ts_code,
300750.SZ,37.6
300803.SZ,28.2
601238.SH,10.0
300379.SZ,3.6
600580.SH,3.3
300290.SZ,3.1
002595.SZ,3.1
000935.SZ,3.0
300010.SZ,3.0


## Summary and caveats

- **Gamma:** where total dealer GEX sits relative to the flip tells whether option hedging is currently damping or amplifying index moves, under the dealer-sign assumption above.
- **Futures:** the regime and top-20 net show whether a down day was met by new hedges (OI up) or long liquidation (OI down); a widening annualised discount means hedging demand exceeds arbitrage capacity.
- **Margin:** the curve is a concentration estimate - how much financing debt sits within x% of its forced-sale price - not a list of exact stop prices.
- None of these is validated yet; plan 09's event study will test whether index moves accelerate through these zones before any of it is used as a signal.

In [12]:
catalog.close()